In [6]:
"""
Local Sensitivity Analysis — Tornado Plot
==========================================
Baseline MJSP = $22.05  (adjust X_LABEL and BASELINE_MJSP below as needed)

Bar shading convention:
  DARK  shade  →  bar at LOW  parameter value
  LIGHT shade  →  bar at HIGH parameter value

Y-axis label format:  "Parameter [baseline_value] (unit)"
"""

import matplotlib
matplotlib.use("Agg")          # remove this line if running interactively
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.transforms as mtransforms
import numpy as np

plt.rc('font',family='Arial')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  ← edit here
# ─────────────────────────────────────────────────────────────────────────────
BASELINE_MJSP = 22.05
X_LABEL       = "Minimum Selling Price (USD/gal)"

# Category colors: (dark_shade, light_shade)
#   DARK  = bar when parameter is at its LOW  value
#   LIGHT = bar when parameter is at its HIGH value
COLORS = {
    "EHF":     ("#605752", "#ada39f"),   # deep orange
}
CAT_ORDER = ["Overall", "RCF", "HDO", "EHF", "OP", "MP"]


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# (label, unit, category, param_baseline, param_low, param_high,
#  mjsp_at_low_param, mjsp_at_high_param)
# ─────────────────────────────────────────────────────────────────────────────
data = [
 # ── EHF ────────────────────────────────────────────────────────────────
    ("Conv. glucose to ethanol",    "%",                  "EHF",     95,     80,      95,     23.87, 22.05),
    ("Conv. glucan to glucose",     "%",                  "EHF",     90,      80,      90,      23.45, 22.05),
    ("Saccharification res. time",  "hr",                 "EHF",     60,       48,       72,       22.03, 22.07),
    ("Cofermentation res. time",    "hr",                 "EHF",     36,       28.8,     43.2,     22.04, 22.06),
    ("Conv. xylan to xylose",       "%",                  "EHF",     90,      80,      90,      22.42, 22.05),
    ("Conv. xylose to ethanol",     "%",                  "EHF",     85,     70,      90,      22.22, 21.88),
    ("Cellulase enzyme loading",        "wt%",                "EHF",     2,       1,      5,     21.25, 24.45),
    ("Cellulase price",             "USD/kg",             "EHF",     0.3877,   0.1939,   0.5816,   21.24, 22.86),
]


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def fmt_num(v):
    """Smart formatter: integers as int, floats with minimal significant decimals."""
    try:
        if float(v) == int(float(v)):
            return str(int(float(v)))
    except (TypeError, ValueError, OverflowError):
        return str(v)
    v = float(v)
    if abs(v) >= 100: return f"{v:.0f}"
    if abs(v) >= 10:  return f"{v:.1f}"
    if abs(v) >= 1:   return f"{v:.2f}"
    if abs(v) >= 0.1: return f"{v:.3f}"
    return f"{v:.4f}"


# ─────────────────────────────────────────────────────────────────────────────
# SORT  (descending swing so largest bar appears at the top after invert_yaxis)
# ─────────────────────────────────────────────────────────────────────────────
data_sorted = sorted(data, key=lambda r: abs(r[7] - r[6]), reverse=True)

n  = len(data_sorted)
ys = np.arange(n)

# Y-tick labels: "Parameter [baseline] (unit)"
y_labels = [f"{r[0]} [{fmt_num(r[3])}] ({r[1]})" for r in data_sorted]


# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────
BAR_H = 0.60    # bar height (data units)
LPAD  = 0.06    # gap between bar tip and its value annotation
FS    = 20     # font size for bar-end annotations

fig, ax = plt.subplots(figsize=(10, n * 0.42 + 2.5))

for i, (name, unit, cat, p_base, p_low, p_high, m_low, m_high) in enumerate(data_sorted):
    dark_c, light_c = COLORS[cat]
    y = ys[i]

    # Each half-bar is anchored at the baseline and extends toward m_low / m_high
    w_low  = abs(m_low  - BASELINE_MJSP)
    w_high = abs(m_high - BASELINE_MJSP)
    l_low  = min(m_low,  BASELINE_MJSP)
    l_high = min(m_high, BASELINE_MJSP)

    # Draw the longer bar first so the shorter one is fully visible on top
    bar_specs = sorted(
        [(w_low,  l_low,  dark_c),
         (w_high, l_high, light_c)],
        key=lambda b: b[0], reverse=True
    )
    for w, l, color in bar_specs:
        if w > 1e-6:
            ax.barh(y, w, BAR_H, left=l, color=color,
                    edgecolor='white', linewidth=0.4, zorder=2)

    # ── Bar-end annotations: show the parameter value at each bound ───────
    for m_val, p_val in [(m_low, p_low), (m_high, p_high)]:
        if abs(m_val - BASELINE_MJSP) < 1e-6:
            continue  # zero-length bar → skip label
        if m_val < BASELINE_MJSP:
            ax.text(m_val - LPAD, y, fmt_num(p_val),
                    ha='right', va='center', fontsize=FS, color='#1a1a1a')
        else:
            ax.text(m_val + LPAD, y, fmt_num(p_val),
                    ha='left',  va='center', fontsize=FS, color='#1a1a1a')


# ── Baseline vertical line ────────────────────────────────────────────────
ax.axvline(BASELINE_MJSP, color='black', linewidth=0.2, zorder=5)

# Baseline label: annotate just above the top x-tick using data coords
# (placed at y = -0.5 in the inverted axis, which is just above the first row)
ax.annotate(
    f"${BASELINE_MJSP:.2f}",
    xy=(BASELINE_MJSP, -0.5), xycoords='data',
    xytext=(0, 4), textcoords='offset points',
    ha='center', va='bottom', fontsize=20, fontweight='bold', color='black'
)


# ── Y-axis ────────────────────────────────────────────────────────────────
ax.set_yticks(ys)
ax.set_yticklabels(y_labels, fontsize=20)
ax.invert_yaxis()                          # y=0 (largest swing) goes to top
ax.tick_params(axis='y', length=0, pad=4)
ax.set_ylim(n - 0.5, -0.5)               # restore sensible limits after invert


ax.set_xlabel(X_LABEL, fontsize=20, labelpad=8)
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
ax.tick_params(axis='x', labelsize=20, length=4, width=0.2)

# ── then in the spine section, swap bottom ↔ top ──
for spine in ['right', 'left']:          # remove these two
    ax.spines[spine].set_visible(False)
ax.spines['bottom'].set_visible(False)   # was your axis line, now hidden
ax.spines['top'].set_visible(True)       # new axis line at top
ax.spines['top'].set_linewidth(0.3)


# ── Legend ────────────────────────────────────────────────────────────────
present_cats = {r[2] for r in data}
handles = []
for cat in CAT_ORDER:
    if cat not in present_cats:
        continue
    dc, lc = COLORS[cat]
    handles += [
        mpatches.Patch(facecolor=dc, edgecolor='#888', linewidth=0.5,
                       label=f"{cat} — low"),
        mpatches.Patch(facecolor=lc, edgecolor='#888', linewidth=0.5,
                       label=f"{cat} — high"),
    ]

leg = ax.legend(
    handles=handles, fontsize=7.5, ncol=3, loc='lower right',
    framealpha=0.93, edgecolor='#cccccc',
    title="Dataset  (dark = low param value,  light = high param value)",
    title_fontsize=7.5,
)
leg.get_frame().set_linewidth(0.6)




# ── Layout & Save ─────────────────────────────────────────────────────────
fig.subplots_adjust(left=0.285, right=0.97, top=0.95, bottom=0.05)
plt.rcParams['svg.fonttype'] = 'none'

plt.savefig("tornado_plot_ehf.svg", dpi=600, bbox_inches=None)
#plt.savefig("tornado_plot.pdf", bbox_inches=None)